# Notebook 03 — Filter, Merge & Clean
- Filters OL noise (western authors, comics, nonfiction)
- Merges Goodreads + OL into unified schema
- Enriches with Google Books + OpenLibrary descriptions
- Removes junk books with no description/ratings
- Final dedup + datatype cleanup
- Output: `Data/Clean/merged_non_western_fantasy.json`

In [1]:
# ── Cell 1 — Setup & load ───────────────────────────────────────────────────
import json, re, time, requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

REPO_ROOT  = Path().resolve().parents[1]
RAW_DIR    = REPO_ROOT / 'Data' / 'Raw' / 'non_western_fantasy'
CLEAN_DIR  = REPO_ROOT / 'Data' / 'Clean'
CLEAN_DIR.mkdir(exist_ok=True)

ol_raw = pd.DataFrame(json.load(open(RAW_DIR / 'ol_genre_first.json')))
gr_raw = pd.read_csv(RAW_DIR / 'goodreads_with_descriptions.csv')

print(f'✅ Loaded')
print(f'   OL books:         {len(ol_raw):,}')
print(f'   Goodreads books:  {len(gr_raw):,}')
print(f'   OL columns:  {list(ol_raw.columns)}')
print(f'   GR columns:  {list(gr_raw.columns)}')

✅ Loaded
   OL books:         4,364
   Goodreads books:  2,129
   OL columns:  ['title', 'author', 'year_published', 'cover_url', 'avg_rating', 'num_ratings', 'subjects', 'source', 'source_url', 'source_tag', 'query', 'description']
   GR columns:  ['title', 'author', 'goodreads_url', 'cover_url', 'avg_rating', 'num_ratings', 'year_published', 'shelf', 'description']


In [2]:
# ── Cell 2 — Filter OL noise ────────────────────────────────────────────────
WESTERN_NOISE_AUTHORS = {
    'edgar rice burroughs', 'h. rider haggard', 'rudyard kipling',
    'joseph conrad', 'wilbur smith', 'james michener', 'john buchan',
    'arthur conan doyle', 'h.g. wells', 'jules verne',
    'charlotte perkins gilman', 'george macdonald', 'd. h. lawrence',
    'lucy maud montgomery', 'carlo collodi', 'neil gaiman',
    'george r. r. martin', 'rick riordan', 'j. k. rowling',
    'bram stoker', 'edith nesbit', 'l. frank baum',
    'richard adams', 'roald dahl', 'c. s. lewis',
    'stephen king', 'astrid lindgren', 'bill willingham',
    'mary shelley', 'niccolò machiavelli', 'michael ende',
    'charles kingsley', 'andrew lang', 'dr. seuss', 'lewis carroll',
    'edgar allan poe', 'nathaniel hawthorne', 'mark twain',
    'antoine de saint-exupéry', 'christopher paolini', 'j.r.r. tolkien',
    'mercedes lackey', 'piers anthony', 'mary pope osborne',
    'daisy meadows', 'erin hunter', 'paul theroux', 'harold macgrath',
    'charles de lint', 'james rollins',
}

NOISE_SUBJECTS = {
    'tarzan', 'colonialism', 'imperialism',
    'safari', 'big game hunting', 'missionaries',
}

COMIC_SUBJECTS = {
    'graphic novel', 'graphic novels', 'comics', 'comic book',
    'comic strip', 'sequential art',
}

NONFICTION_SUBJECTS = {
    'criticism', 'critical essays', 'literary criticism',
    'history and criticism', 'congresses', 'study and teaching',
    'nonfiction', 'biography', 'autobiography',
    'education', 'philosophy', 'sociology', 'political science',
    'essays', 'handbooks', 'reference',
    'cross-cultural studies', 'mass media', 'self-perception',
    'creative ability in children', 'imagination in children',
}

def ol_final_filter(row):
    author        = str(row.get('author', '')).lower().strip()
    subjects_text = ' '.join(str(s) for s in (row.get('subjects') or [])).lower()
    if author in WESTERN_NOISE_AUTHORS:
        return False
    if any(n in subjects_text for n in NOISE_SUBJECTS):
        return False
    if any(c in subjects_text for c in COMIC_SUBJECTS):
        return False
    if any(n in subjects_text for n in NONFICTION_SUBJECTS):
        return False
    return True

mask       = ol_raw.apply(ol_final_filter, axis=1)
ol_clean   = ol_raw[mask].copy().reset_index(drop=True)

# Drop blank authors
ol_clean = ol_clean[ol_clean['author'].str.strip() != ''].copy().reset_index(drop=True)

print(f'── OL filter ──')
print(f'  Before:  {len(ol_raw):,}')
print(f'  After:   {len(ol_clean):,}')
print(f'  Removed: {len(ol_raw) - len(ol_clean):,}')

── OL filter ──
  Before:  4,364
  After:   2,256
  Removed: 2,108


In [3]:
# ── Cell 3 — Normalize + merge ──────────────────────────────────────────────
def norm_title(t):
    t = str(t).lower().strip()
    t = re.sub(r'\s*[\(\[].*?[\)\]]\s*', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def norm_author(a):
    return str(a).lower().strip()

ol_unified = pd.DataFrame({
    'title':          ol_clean['title'].str.strip(),
    'author':         ol_clean['author'].fillna(''),
    'description':    ol_clean['description'].fillna(''),
    'year_published': ol_clean['year_published'],
    'cover_url':      ol_clean['cover_url'].fillna(''),
    'avg_rating':     ol_clean['avg_rating'],
    'num_ratings':    ol_clean['num_ratings'],
    'subjects':       ol_clean['subjects'].apply(lambda x: x if isinstance(x, list) else []),
    'source':         'open_library',
    'source_url':     ol_clean['source_url'],
    'source_tag':     ol_clean['source_tag'],
    'title_norm':     ol_clean['title'].apply(norm_title),
    'author_norm':    ol_clean['author'].fillna('').apply(norm_author),
})

gr_unified = pd.DataFrame({
    'title':          gr_raw['title'].str.strip(),
    'author':         gr_raw['author'].fillna(''),
    'description':    gr_raw['description'].fillna(''),
    'year_published': gr_raw['year_published'],
    'cover_url':      gr_raw['cover_url'].fillna(''),
    'avg_rating':     gr_raw['avg_rating'],
    'num_ratings':    gr_raw['num_ratings'],
    'subjects':       [[] for _ in range(len(gr_raw))],
    'source':         'goodreads',
    'source_url':     gr_raw['goodreads_url'],
    'source_tag':     gr_raw['shelf'],
    'title_norm':     gr_raw['title'].apply(norm_title),
    'author_norm':    gr_raw['author'].fillna('').apply(norm_author),
})

# GR wins on overlap
gr_lookup = {}
for _, row in gr_unified.iterrows():
    gr_lookup[(row['title_norm'], row['author_norm'])] = row

enriched_ol = []
ol_matched  = 0
for _, row in ol_unified.iterrows():
    key      = (row['title_norm'], row['author_norm'])
    gr_match = gr_lookup.get(key)
    if gr_match is not None:
        ol_matched += 1
        row = row.copy()
        if gr_match['description']:    row['description']    = gr_match['description']
        if gr_match['cover_url']:      row['cover_url']      = gr_match['cover_url']
        if pd.notna(gr_match['avg_rating']):   row['avg_rating']   = gr_match['avg_rating']
        if pd.notna(gr_match['num_ratings']):  row['num_ratings']  = gr_match['num_ratings']
        if pd.notna(gr_match['year_published']): row['year_published'] = gr_match['year_published']
    enriched_ol.append(row)

ol_enriched = pd.DataFrame(enriched_ol)
gr_keys     = set(gr_lookup.keys())
ol_only     = ol_enriched[~ol_enriched.apply(lambda r: (r['title_norm'], r['author_norm']) in gr_keys, axis=1)]
merged      = pd.concat([gr_unified, ol_only], ignore_index=True)
merged      = merged.drop(columns=['title_norm', 'author_norm'])

print(f'── Merge results ──')
print(f'  Goodreads:       {len(gr_unified):,}')
print(f'  OL matched GR:   {ol_matched:,}')
print(f'  OL-only added:   {len(ol_only):,}')
print(f'  Total:           {len(merged):,}')

── Merge results ──
  Goodreads:       2,129
  OL matched GR:   103
  OL-only added:   2,153
  Total:           4,282


In [4]:
# ── Cell 4 — Google Books enrichment ───────────────────────────────────────
import os
from dotenv import load_dotenv
from langdetect import detect

load_dotenv()
GOOGLE_BOOKS_API_KEY = os.getenv('GOOGLE_BOOKS_API_KEY', '')
GB_CKPT_PATH = CLEAN_DIR / 'google_books_checkpoint.json'
print(f'API key loaded: {"✅" if GOOGLE_BOOKS_API_KEY else "❌ missing"}')

def fetch_google_books_description(title, author=''):
    query  = f'{title} {author}'.strip()
    params = {
        'q': query, 'maxResults': 5, 'key': GOOGLE_BOOKS_API_KEY,
        'fields': 'items(volumeInfo(title,authors,description,publishedDate,imageLinks))'
    }
    try:
        resp  = requests.get('https://www.googleapis.com/books/v1/volumes', params=params, timeout=10)
        resp.raise_for_status()
        for item in resp.json().get('items', []):
            info = item.get('volumeInfo', {})
            desc = info.get('description', '')
            if desc:
                try:
                    if detect(desc) == 'en':
                        return {'description': desc, 'cover_url': info.get('imageLinks', {}).get('thumbnail', '')}
                except: continue
        return None
    except: return None

gb_results = json.load(open(GB_CKPT_PATH)) if GB_CKPT_PATH.exists() else {}
needs_desc = merged[merged['description'].apply(lambda x: not bool(str(x).strip()) or str(x).strip() == 'nan')]
remaining  = needs_desc[~needs_desc['source_url'].isin(gb_results.keys())]
print(f'Resuming — {len(gb_results):,} done | {len(remaining):,} remaining')

stats = {'filled': 0, 'not_found': 0}
for i, (idx, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining), desc='Google Books'), 1):
    result = fetch_google_books_description(row['title'], row['author'])
    gb_results[row['source_url']] = result if result else {}
    stats['filled' if result else 'not_found'] += 1
    if i % 100 == 0:
        json.dump(gb_results, open(GB_CKPT_PATH, 'w'))
    time.sleep(0.5)

json.dump(gb_results, open(GB_CKPT_PATH, 'w'))
print(f'✅ Done — filled: {stats["filled"]:,} | not found: {stats["not_found"]:,}')

# Merge back
for idx, row in merged.iterrows():
    gb = gb_results.get(row['source_url'], {})
    if gb.get('description') and not bool(str(row['description']).strip()):
        merged.at[idx, 'description'] = gb['description']
    if gb.get('cover_url') and not bool(str(row['cover_url']).strip()):
        merged.at[idx, 'cover_url'] = gb['cover_url']

API key loaded: ✅
Resuming — 2,174 done | 0 remaining


Google Books: 0it [00:00, ?it/s]

✅ Done — filled: 0 | not found: 0


In [5]:
# ── Cell 5 — OpenLibrary description enrichment ────────────────────────────
OL_DESC_CKPT = CLEAN_DIR / 'ol_desc_checkpoint.json'

def fetch_ol_description(ol_url):
    key = ol_url.replace('https://openlibrary.org', '')
    if not key.startswith('/works/'): return None
    try:
        r    = requests.get(f'https://openlibrary.org{key}.json', timeout=10)
        desc = r.json().get('description', '')
        if isinstance(desc, dict): desc = desc.get('value', '')
        return str(desc).strip() if desc and len(str(desc).strip()) > 20 else None
    except: return None

ol_desc_results = json.load(open(OL_DESC_CKPT)) if OL_DESC_CKPT.exists() else {}
still_missing   = merged[
    merged['description'].apply(lambda x: not bool(str(x).strip()) or str(x).strip() == 'nan') &
    (merged['source'] == 'open_library')
]
remaining = still_missing[~still_missing['source_url'].isin(ol_desc_results.keys())]
print(f'OL desc — {len(ol_desc_results):,} done | {len(remaining):,} remaining')

stats = {'filled': 0, 'not_found': 0}
for i, (idx, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining), desc='OL descriptions'), 1):
    result = fetch_ol_description(row['source_url'])
    ol_desc_results[row['source_url']] = result or ''
    stats['filled' if result else 'not_found'] += 1
    if i % 100 == 0: json.dump(ol_desc_results, open(OL_DESC_CKPT, 'w'))
    time.sleep(0.3)

json.dump(ol_desc_results, open(OL_DESC_CKPT, 'w'))
print(f'✅ Done — filled: {stats["filled"]:,} | not found: {stats["not_found"]:,}')

for idx, row in merged.iterrows():
    desc = ol_desc_results.get(row['source_url'], '')
    if desc and not bool(str(row['description']).strip()):
        merged.at[idx, 'description'] = desc

OL desc — 1,156 done | 0 remaining


OL descriptions: 0it [00:00, ?it/s]

✅ Done — filled: 0 | not found: 0


In [6]:
# ── Cell 6 — Remove junk + subjects fallback ───────────────────────────────
JUNK_SIGNALS = [
    'tagebuch', 'notizbuch', 'malbuch', 'planer', 'kalender',
    'coloring', 'colouring', 'color by number', 'weight tracker',
    'expense tracker', 'password log', 'music sheet', 'genkoyoushi',
    'house sitting', 'self care', 'reading list book',
    'choreograph', 'dancing for', 'folk dance', 'cookbook', 'recipes',
    'guitar solo', 'diabetes', 'trivia', 'food fantasy fgb',
]

def is_junk(row):
    has_desc    = bool(str(row.get('description', '')).strip()) and str(row.get('description','')).strip() != 'nan'
    has_ratings = pd.notna(row.get('num_ratings')) and row.get('num_ratings', 0) > 0
    if has_desc or has_ratings: return False
    return any(s in str(row.get('title', '')).lower() for s in JUNK_SIGNALS)

merged = merged[~merged.apply(is_junk, axis=1)].copy().reset_index(drop=True)

# Subjects fallback for remaining missing
for idx, row in merged.iterrows():
    if bool(str(row['description']).strip()) and str(row['description']).strip() != 'nan': continue
    subj = [str(s).strip() for s in (row['subjects'] if isinstance(row['subjects'], list) else []) if len(str(s).strip()) > 2][:10]
    if subj: merged.at[idx, 'description'] = 'A work of fantasy fiction involving: ' + ', '.join(subj) + '.'

still_missing = merged[merged['description'].apply(lambda x: not bool(str(x).strip()) or str(x).strip() == 'nan')]
print(f'After junk removal + fallback:')
print(f'  Total: {len(merged):,}')
print(f'  Still missing desc: {len(still_missing):,}')

After junk removal + fallback:
  Total: 4,193
  Still missing desc: 144


In [7]:
REMOVE_AUTHORS_FINAL = {
    'stephen king', 'neil gaiman', 'kazuo ishiguro',
    'truman capote', 'shel silverstein', 'laura dave',
    'gabrielle zevin', 'marissa meyer', 'pierce brown',
    'lynne reid banks', 'gillian rubinstein', 'naomi novik',
    'diana wynne jones', 'raymond e. feist', 'andre norton',
    'alastair reynolds', 'kate elliott', 'guy gavriel kay',
    'catherynne m. valente', 'lafcadio hearn', 'talbot mundy',
    'luis senarens', 'tracey west', 'square enix',
    'hasaway anime corner', 'erica baker', 'reggie robinson',
    'tom deitz', 'scott walker',
    # Final batch
    'maya daniels', 'tara maya', 'carmen kern', 'henriette barkow',
    "stephen king", "caroline peckham", "neal stephenson", 
    "sarah a. parker", "samantha shannon", "olivie blake",
    "don freeman",
}


before = len(merged)
merged = merged[~merged['author'].str.lower().str.strip().isin(REMOVE_AUTHORS_FINAL)].copy().reset_index(drop=True)
print(f'Western authors removed: {before - len(merged):,}')
print(f'Remaining: {len(merged):,}')

Western authors removed: 134
Remaining: 4,059


In [8]:
# ── Cell 8 — Dedup + datatype cleanup + final save ─────────────────────────
import re

def clean_title(title):
    title = re.sub(r'\s*[\(\[](hardcover|paperback|kindle edition|ebook|mass market paperback|audio cd|audiobook|unknown binding|library binding|board book|large print)[\)\]]',
                   '', title, flags=re.IGNORECASE)
    return title.strip()

# Dedup
before = len(merged)
merged['_title_norm']  = merged['title'].str.lower().str.strip()
merged['_author_norm'] = merged['author'].str.lower().str.strip()
merged['_source_rank'] = merged['source'].map({'goodreads': 0, 'open_library': 1})
merged = merged.sort_values('_source_rank').drop_duplicates(
    subset=['_title_norm', '_author_norm'], keep='first'
).drop(columns=['_title_norm', '_author_norm', '_source_rank']).reset_index(drop=True)
print(f'Duplicates removed: {before - len(merged):,}')

# Clean titles
merged['title'] = merged['title'].apply(clean_title)
print(f'Titles cleaned')

# Datatypes
merged['year_published'] = pd.to_numeric(merged['year_published'], errors='coerce').fillna(0).astype(int).replace(0, None)
merged['num_ratings']    = pd.to_numeric(merged['num_ratings'], errors='coerce').fillna(0).astype(int)
merged['avg_rating']     = merged['avg_rating'].round(2)
merged['subjects']       = merged['subjects'].apply(lambda x: x if isinstance(x, list) else [])
for col in ['title', 'author', 'description', 'cover_url', 'source_url', 'source_tag']:
    merged[col] = merged[col].str.strip()

# Save
FINAL_PATH = CLEAN_DIR / 'merged_non_western_fantasy.json'
merged.to_json(FINAL_PATH, orient='records', indent=2, force_ascii=False)
merged.to_csv(CLEAN_DIR / 'merged_non_western_fantasy.csv', index=False)

print(f'\n✅ NB03 complete!')
print(f'   Total books: {len(merged):,}')
print(f'   Saved to: {FINAL_PATH}')

Duplicates removed: 64
Titles cleaned

✅ NB03 complete!
   Total books: 3,995
   Saved to: C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\Data\Clean\merged_non_western_fantasy.json


In [9]:
# ── Quick final author scan ─────────────────────────────────────────────────
author_counts = merged["author"].value_counts()
print("Authors with 5+ books:")
print(author_counts[author_counts >= 5].to_string())

Authors with 5+ books:
author
Nnedi Okorafor                43
N.K. Jemisin                  26
Octavia E. Butler             23
Samuel R. Delany              21
Mò Xiāng Tóng Xiù             19
Nalo Hopkinson                17
Tao Wong                      16
Joseph Li                     13
Ta-Nehisi Coates              12
P. Djèlí Clark                11
Brandon  Thomas               11
Lian Hearn                    10
Roshani Chokshi               10
Will Wight                    10
Luo Di Cheng Qiu              10
Tananarive Due                 9
Nghi Vo                        9
Chloe Gong                     9
J.C. Kang                      9
Victor LaValle                 9
Aliette de Bodard              8
Sheree Renée Thomas            8
L. A. Banks                    8
村上春樹                           8
Nisi Shawl                     7
R.F. Kuang                     7
Elizabeth Lim                  7
Fonda Lee                      7
Yoon Ha Lee                    7
Anastasis Bly

In [10]:
# ── Fix + remove ───────────────────────────────────────────────────────────
before = len(merged)

# Normalize whitespace in author column first
merged["author"] = merged["author"].str.replace(r'\s+', ' ', regex=True).str.strip()

# Now remove
EXTRA_REMOVE = {
    "stephen king", "caroline peckham",
}

merged = merged[~merged["author"].str.lower().str.strip().isin(EXTRA_REMOVE)].copy().reset_index(drop=True)
print(f"Removed: {before - len(merged):,}")
print(f"Remaining: {len(merged):,}")

merged["subjects"] = merged["subjects"].apply(lambda x: x if isinstance(x, list) else [])
merged.to_json(CLEAN_DIR / "merged_non_western_fantasy.json", orient="records", indent=2, force_ascii=False)
merged.to_csv(CLEAN_DIR / "merged_non_western_fantasy.csv", index=False)
print("✅ Saved")

Removed: 1
Remaining: 3,994
✅ Saved


In [11]:
# ── Check what LaGuardia looks like in the data ────────────────────────────
laguardia = merged[merged["title"].str.contains("LaGuardia", case=False, na=False)]
for _, r in laguardia.iterrows():
    print(f"Title: {r['title']}")
    print(f"Author: {r['author']}")
    print(f"Source tag: {r['source_tag']}")
    print(f"Subjects: {r['subjects']}")
    print(f"Description: {r['description'][:200]}")
    print()

Title: LaGuardia
Author: Nnedi Okorafor
Source tag: afrofuturism
Subjects: []
Description: From Hugo, Nebula, and World Fantasy Award Winner Nnedi Okorafor (Who Fears Death, Binti, Akata series) comes Laguardia. Set in an alternative world where aliens have come to Earth and integrated with

Title: LaGuardia #1
Author: Nnedi Okorafor
Source tag: afrofuturism
Subjects: []
Description: Set in an alternative world where aliens have come to Earth and integrated with society, LaGuardia revolves around a pregnant Nigerian-American doctor, Future Nwafor Chukwuebuka who has just returned 

Title: LaGuardia #2
Author: Nnedi Okorafor
Source tag: afrofuturism
Subjects: []
Description: As Future Nwafor Chukwuebuka fights to save the life of her beloved illegal- alien plant, Letme Live, a group of immigrants are held captive at LaGuardia International and Interstellar Airport because



In [12]:
# Remove issue numbers from titles for dedup purposes
# e.g. "LaGuardia #1" and "LaGuardia" are the same
merged["title_dedup"] = merged["title"].str.replace(r'\s*#\d+$', '', regex=True).str.strip()
merged = merged.drop_duplicates(subset=["title_dedup", "author"], keep="first")
merged = merged.drop(columns=["title_dedup"])
print(f"After dedup: {len(merged):,}")

After dedup: 3,973


In [16]:
merged["subjects"] = merged["subjects"].apply(lambda x: x if isinstance(x, list) else [])
merged.to_json(CLEAN_DIR / "merged_non_western_fantasy.json", orient="records", indent=2, force_ascii=False)
merged.to_csv(CLEAN_DIR / "merged_non_western_fantasy.csv", index=False)
print(f"✅ Saved — {len(merged):,} books")

✅ Saved — 3,973 books


In [14]:
print(f"Current merged: {len(merged):,}")

Current merged: 3,973
